Creating two clustering models using data curated by StatCan.

In [ ]:
!python -m pip install "numpy<2.0" "pandas<2.0"
!python -m pip install scikit-learn==1.6.0
!python -m pip install matplotlib==3.9.3
!python -m pip install hdbscan==0.8.40
!python -m pip install geopandas==1.0.1
!python -m pip install contextily==1.6.2
!python -m pip install shapely==2.0.6

In [5]:
import importlib.util as u
print({p: u.find_spec(p) is not None for p in ["numpy","pandas","sklearn","matplotlib", "shapely", "contextily", "geopandas", "hdbscan"]})

{'numpy': True, 'pandas': True, 'sklearn': True, 'matplotlib': True, 'shapely': True, 'contextily': True, 'geopandas': True, 'hdbscan': True}


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import hdbscan
from sklearn.preprocessing import StandardScaler

# geographical tools
import geopandas as gpd  # pandas dataframe-like geodataframes for geographical data
import contextily as ctx  # used for obtianing a basemap of Canada
from shapely.geometry import Point

import warnings
warnings.filterwarnings('ignore')

In [8]:
# downloading canada map

import requests
import zipfile
import io
import os

zip_file_url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/YcUk-ytgrPkmvZAh5bf7zA/Canada.zip'

# Directory to save the extracted TIFF file
output_dir = './'
os.makedirs(output_dir, exist_ok=True)

# Step 1: Download the ZIP file
response = requests.get(zip_file_url)
response.raise_for_status()  # Ensure the request was successful
# Step 2: Open the ZIP file in memory
with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
    # Step 3: iterating over the files in the ZIP
    for file_name in zip_ref.namelist():
        if file_name.endswith('.tif'):  # Check if it's a TIFF file
            # Step 4: extracting the TIFF file
            zip_ref.extract(file_name, output_dir)
            print(f"Downloaded and extracted: {file_name}")

Downloaded and extracted: Canada.tif


In [12]:
# plotting my data points as geographic locations on top of a Canada map

import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

def plot_clustered_locations(df, title='Museums Clustered by Proximity'):


    # Check required columns
    required_cols = {'Latitude', 'Longitude', 'Cluster'}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"DataFrame must contain columns: {required_cols}")

    # Load coordinates into a GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs="EPSG:4326"
    )

    # Reproject to Web Mercator
    gdf = gdf.to_crs(epsg=3857)

    # Create plot
    fig, ax = plt.subplots(figsize=(15, 10))

    # Separate clustered points from noise
    non_noise = gdf[gdf['Cluster'] != -1]
    noise = gdf[gdf['Cluster'] == -1]

    # Plot noise points
    if not noise.empty:
        noise.plot(ax=ax, color='k', markersize=30, edgecolor='r', alpha=1, label='Noise')

    # Plot clustered points
    if not non_noise.empty:
        non_noise.plot(
            ax=ax,
            column='Cluster',
            cmap='tab10',
            markersize=30,
            edgecolor='k',
            legend=False,
            alpha=0.6
        )

    # Add basemap
    ctx.add_basemap(ax, source='./Canada.tif')

    # Format
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()